# 04 回归训练并在样本外预测

图片沿用二分类那一版，不重新生成。`WINDOW_KEY = "I20"` 读 `(N, 3, 100, 60)`，`"I60"` 读 `(N, 3, 100, 180)`。

目标是 `labels.parquet` 里的 `future_ret`：个股 5 日收益减去 688 平权 5 日收益。网络输出 1 个数，损失是均方误差。学习率固定 `1e-5`，没有衰减，也没有早停。每个种子固定跑 10 个 epoch，留下验证集 mse 最低的那一轮。

日志里的 `dir` 是方向正确率：预测超额大于 0 是否和真实超额同号。它只打印，不决定留哪一轮。

图片：`/storage/server/144server/cq/ChenSiwei/k_pictures_rgb_I20/{I20|I60}/`。
权重和预测写到旁边，不覆盖二分类：`/storage/server/144server/cq/ChenSiwei/k_pictures_rgb_I20_reg/`。

和正在跑的二分类同时读这份图，两边都会更慢。先 `N_SEEDS = 1` 试通。

In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
candidates = [cwd, cwd / "change_rgb_i20"]
candidates.extend(cwd.parents)
for parent in [cwd, *cwd.parents]:
    candidates.append(parent / "change_rgb_i20")
seen = set()
root = None
for p in candidates:
    key = str(p)
    if key in seen:
        continue
    seen.add(key)
    if (p / "helpers.py").exists() and (p / "config.yaml").exists() and (p / "train_lib.py").exists():
        text = (p / "config.yaml").read_text(encoding="utf-8")
        if "task: regression" in text:
            root = p
            break
if root is None:
    raise FileNotFoundError(
        f"找不到 change_rgb_i20（config.yaml 里要有 task: regression）。cwd={cwd}"
    )
missing = [name for name in ("cnn.py", "train_lib.py", "backtest_lib.py") if not (root / name).exists()]
if missing:
    raise FileNotFoundError(
        f"{root} 缺少 {missing}。这些文件和 helpers.py 放在同一目录，不要再套一层。"
    )
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import torch
from helpers import notebook_bootstrap
from train_lib import predict_window, train_window

ROOT, cfg, paths = notebook_bootstrap()
print("ROOT   =", ROOT)
print("task   =", cfg.cnn.task, "epochs =", cfg.cnn.max_epochs, "lr =", cfg.cnn.optimizer.lr)
print("images =", cfg.paths.images_remote)
print("output =", cfg.paths.output_remote)
print("cuda   =", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 训练

`SKIP_EXISTING=True` 时已有最终的 `seed_k.pt` 会跳过。每个 epoch 会写 `seed_k.running.pt`，里面是到目前为止验证集 mse 最低的权重。10 轮全部跑完后，`seed_k.pt` 才是最终留下的那一轮，`running` 文件会删掉。

`WINDOW_KEY` 用 `"I20"` 或 `"I60"`，须与已经生成的图一致。这里改的是读哪一套图，不是重新画图。

In [ ]:
WINDOW_KEY = "I20"   # 或 "I60"，须与已生成的图一致
N_SEEDS = 1
SKIP_EXISTING = True

results = train_window(
    cfg, paths, WINDOW_KEY,
    n_seeds=N_SEEDS,
    num_workers=0,
    skip_existing=SKIP_EXISTING,
    pin_memory=False,
)
print(results)

## 样本外预测（test split，已有种子平均）

`pred` 是预测的 5 日超额。多个种子先各自还原成超额，再取平均。试通后把上面的 `N_SEEDS` 改成 5，再跑训练和这一格。然后打开 `05_plot_top10.ipynb`。

In [ ]:
pred = predict_window(cfg, paths, WINDOW_KEY, num_workers=0)
display(pred.head())